# Binary Task — `PROCEDENTE` vs `IMPROCEDENTE`

A deliberately simpler diagnostic task, orthogonal to `22`-`25`: instead of trying to recover all 10 (or 8/6/5) outcome labels.

## Section 1 — Setup & Load

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, str(Path('../..').resolve()))
from src.config import DATA_DIR

df = pd.read_csv(DATA_DIR / 'preprocessed_grupo3.csv')
print(f'Loaded {len(df):,} rows, {df["Decisao"].nunique()} classes')
df['Decisao'].value_counts()

## Section 2 — Filter to the Two Classes

No relabeling.

In [ ]:
KEEP_CLASSES = ['PROCEDENTE', 'IMPROCEDENTE']

before_n = len(df)
df = df[df['Decisao'].isin(KEEP_CLASSES)].reset_index(drop=True)
after_n = len(df)
print(f'Rows: {before_n:,} -> {after_n:,} ({(before_n - after_n) / before_n:.1%} dropped)')
df['Decisao'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
df['Decisao'].value_counts().sort_values().plot.barh(ax=ax, color='#6A1B9A')
ax.set_title(f'Binary task — {len(df):,} rows')
ax.set_xlabel('Documents')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('../../reports/figures/class_distribution_binary_pi.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 3 — Sanity Checks

In [ ]:
assert set(df['split'].unique()) <= {'train', 'val', 'test'}

split_counts = df.groupby(['split', 'Decisao']).size().unstack('Decisao').fillna(0).astype(int)
split_counts = split_counts.reindex(['train', 'val', 'test'])
print('Rows per class, per split:')
print(split_counts.T.to_string())

train_classes = set(df.loc[df['split'] == 'train', 'Decisao'].unique())
for other_split in ['val', 'test']:
    split_classes = set(df.loc[df['split'] == other_split, 'Decisao'].unique())
    orphans = train_classes - split_classes
    print(f'\nTrain classes absent from {other_split}: {len(orphans)}')
    assert not orphans, f'{orphans} has zero rows in {other_split} -- cannot be evaluated'

## Section 4 — Export

Written as `preprocessed_grupo3_binary_pi.csv` + `label2id_grupo3_binary_pi.json`.

In [ ]:
EXPORT_COLS = [
    'ImportJurisprudenciaId', 'source', 'Sumario_raw', 'Sumario', 'Decisao',
    'TipoDecisao', 'group_id', 'split', 'cv_fold',
]

export_path = DATA_DIR / 'preprocessed_grupo3_binary_pi.csv'
df[EXPORT_COLS].to_csv(export_path, index=False, encoding='utf-8')
print(f'Exported: {export_path}  ({len(df):,} rows)')

le = LabelEncoder().fit(df['Decisao'])
labels_bin = list(le.classes_)
label2id_bin = {cls: int(i) for i, cls in enumerate(le.classes_)}
id2label_bin = {int(i): cls for i, cls in enumerate(le.classes_)}

label_path = DATA_DIR / 'label2id_grupo3_binary_pi.json'
with open(label_path, 'w', encoding='utf-8') as f:
    json.dump(
        {'labels': labels_bin, 'label2id': label2id_bin, 'id2label': id2label_bin},
        f, ensure_ascii=False, indent=2,
    )
print(f'Label mapping saved -> {label_path}  ({len(label2id_bin)} classes)')